In [57]:
import pandas as pd
import numpy as np

In [58]:
df = pd.read_csv("../data/ndvi_dataset.csv")

ndvi_cols = [
    col for col in df.columns
    if col.startswith("ndvi_")
]

print("Dataset shape:", df.shape)
print("Number of NDVI columns:", len(ndvi_cols))

Dataset shape: (908, 27)
Number of NDVI columns: 24


In [59]:
ndvi = df[ndvi_cols]

features = pd.DataFrame(index=df.index)

features["ndvi_mean"] = ndvi.mean(axis=1)

features["ndvi_std"] = ndvi.std(axis=1)

features["ndvi_min"] = ndvi.min(axis=1)

features["ndvi_max"] = ndvi.max(axis=1)

features["ndvi_range"] = (
    features["ndvi_max"] -
    features["ndvi_min"]
)

features["ndvi_first"] = ndvi.iloc[:, 0]

features["ndvi_last"] = ndvi.iloc[:, -1]

features["overall_change"] = (
    features["ndvi_last"] -
    features["ndvi_first"]
)

features["largest_drop"] = (
    ndvi.diff(axis=1).min(axis=1)
)

time = np.arange(len(ndvi_cols))

features["ndvi_slope"] = ndvi.apply(
    lambda row: np.polyfit(
        time,
        row.values,
        1
    )[0],
    axis=1
)

midpoint = len(ndvi_cols) // 2

features["early_mean"] = (
    ndvi.iloc[:, :midpoint].mean(axis=1)
)

features["late_mean"] = (
    ndvi.iloc[:, midpoint:].mean(axis=1)
)

features["early_late_change"] = (
    features["late_mean"] -
    features["early_mean"]
)

features["min_ndvi_timestep"] = (
    ndvi.values.argmin(axis=1)
)

In [64]:
featured_df = pd.concat(
    [
        df[["lon", "lat", "label"]],
        features
    ],
    axis=1
)

print("Feature-engineered shape:", featured_df.shape)
print(featured_df.columns.tolist())
featured_df.head()
print(featured_df.isnull().sum())
features.describe()

Feature-engineered shape: (908, 17)
['lon', 'lat', 'label', 'ndvi_mean', 'ndvi_std', 'ndvi_min', 'ndvi_max', 'ndvi_range', 'ndvi_first', 'ndvi_last', 'overall_change', 'largest_drop', 'ndvi_slope', 'early_mean', 'late_mean', 'early_late_change', 'min_ndvi_timestep']
lon                  0
lat                  0
label                0
ndvi_mean            0
ndvi_std             0
ndvi_min             0
ndvi_max             0
ndvi_range           0
ndvi_first           0
ndvi_last            0
overall_change       0
largest_drop         0
ndvi_slope           0
early_mean           0
late_mean            0
early_late_change    0
min_ndvi_timestep    0
dtype: int64


,ndvi_mean,ndvi_std,ndvi_min,ndvi_max,ndvi_range,ndvi_first,ndvi_last,overall_change,largest_drop,ndvi_slope,early_mean,late_mean,early_late_change,min_ndvi_timestep
count,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000,908.000000
mean,0.600447,0.138287,0.372313,0.831073,0.458760,0.812217,0.502530,-0.309687,-0.179247,-0.013967,0.694863,0.506030,-0.188833,15.604626
std,0.082122,0.034855,0.101143,0.085862,0.110392,0.107254,0.083920,0.096654,0.098312,0.006398,0.116647,0.066571,0.095397,5.186814
min,0.225397,0.019145,-0.090503,0.261597,0.064525,0.244101,0.173526,-0.636498,-0.680752,-0.031160,0.223903,0.226890,-0.433798,1.000000
25%,0.543273,0.116439,0.311071,0.799755,0.384889,0.782659,0.465174,-0.355687,-0.218242,-0.018401,0.620337,0.462050,-0.254658,13.000000
50%,0.621853,0.139938,0.371447,0.866741,0.447734,0.859838,0.522110,-0.319810,-0.160148,-0.015246,0.730543,0.510784,-0.203505,17.000000
75%,0.663412,0.160869,0.445004,0.885498,0.524467,0.879048,0.560783,-0.266469,-0.111934,-0.010117,0.785441,0.553699,-0.135440,19.000000
max,0.748969,0.255715,0.611776,0.927007,0.936429,0.917479,0.642817,0.102552,-0.035810,0.015043,0.866689,0.678604,0.243199,23.000000


In [65]:
feature_cols = [
    "ndvi_mean",
    "ndvi_std",
    "ndvi_min",
    "ndvi_max",
    "ndvi_range",
    "ndvi_first",
    "ndvi_last",
    "overall_change",
    "largest_drop",
    "ndvi_slope",
    "early_mean",
    "late_mean",
    "early_late_change",
    "min_ndvi_timestep"
]

X = featured_df[feature_cols]

y = featured_df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (908, 14)
y shape: (908,)


In [66]:
featured_df.to_csv(
    "../data/ndvi_features.csv",
    index=False
)

print("Feature dataset saved successfully.")

Feature dataset saved successfully.
